Few parts of the code was enhanced using Generative AIs.

In [ ]:
!pip install nibabel

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 14.3 MB/s eta 0:00:00


In [ ]:
import os
import zipfile
import shutil
import glob
import nibabel as nib
import numpy as np
from sklearn.model_selection import train_test_split

# Define the source and destination paths for both datasets

source_dir_brats_2017 = '/content/drive/MyDrive/ColabNotebooks/NewDataset/Brats17TrainingData'
brats_2017_dir = '/content/drive/MyDrive/ColabNotebooks/NewDataset/dsihe_MICCAI_BraTS_2017'

source_dir_brats_2019 = '/content/drive/MyDrive/ColabNotebooks/NewDataset/MICCAI_BraTS_2019_Data_Training'
brats_2019_dir = '/content/drive/MyDrive/ColabNotebooks/NewDataset/dsihe_MICCAI_BraTS_2019'

output_dir = '/content/drive/MyDrive/ColabNotebooks/NewDataset/data_split'

# List of file patterns to extract
file_patterns_dsihe = ['_flair_LPS_rSRI.nii.gz', '_t1_LPS_rSRI.nii.gz', '_t1ce_LPS_rSRI.nii.gz', '_t2_LPS_rSRI.nii.gz']
file_patterns_brats = ['_flair.nii.gz', '_t1.nii.gz', '_t1ce.nii.gz', '_t2.nii.gz', '_seg.nii.gz']

# Function to extract files from zip
def extract_files_from_zip(zip_path, dest_path, file_patterns):
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        for file in zip_ref.namelist():
            if any(pattern in file for pattern in file_patterns):
                zip_ref.extract(file, dest_path)
                # Move the file to the correct sub-directory structure
                file_path = os.path.join(dest_path, file)
                dest_file_path = os.path.join(dest_path, os.path.basename(file))
                shutil.move(file_path, dest_file_path)
                # Clean up empty directories
                try:
                    os.removedirs(os.path.dirname(file_path))
                except OSError:
                    pass

# Function to traverse directories and extract files
def traverse_and_extract(source, destination, file_patterns):
    for root, dirs, files in os.walk(source):
        for file in files:
            if file.endswith('.zip'):
                zip_path = os.path.join(root, file)
                rel_path = os.path.relpath(root, source)
                dest_path = os.path.join(destination, rel_path)
                os.makedirs(dest_path, exist_ok=True)
                extract_files_from_zip(zip_path, dest_path, file_patterns)
            elif any(pattern in file for pattern in file_patterns):
                file_path = os.path.join(root, file)
                rel_path = os.path.relpath(root, source)
                dest_path = os.path.join(destination, rel_path)
                os.makedirs(dest_path, exist_ok=True)
                shutil.copy(file_path, dest_path)

# Function to load a NIfTI file
def load_nifti_file(file_path):
    nifti_image = nib.load(file_path)
    image_data = nifti_image.get_fdata()
    return image_data

# Function to save a NIfTI file
def save_nifti_file(data, file_path):
    nifti_image = nib.Nifti1Image(data, np.eye(4))
    nib.save(nifti_image, file_path)

# Function to get patient data from a directory (handles both DSIHE and BRATS)
def get_patient_data(directory, patient_id, dataset_type, tumor_class=None):
    data = {}
    if dataset_type == 'dsihe':
        modalities = ['flair', 't1', 't1ce', 't2']
        for modality in modalities:
            modality_path_pattern = os.path.join(directory, f"dsihe_{patient_id}_*_{modality}_LPS_rSRI.nii.gz")
            modality_files = glob.glob(modality_path_pattern)
            if len(modality_files) == 0:
                print(f"Skipping DSIHE patient {patient_id} due to missing modality: {modality}")
                return None
            if len(modality_files) > 1:
                print(f"Skipping DSIHE patient {patient_id} due to multiple files found for modality: {modality}")
                return None
            data[modality] = load_nifti_file(modality_files[0])
        patient_parts = patient_id.split('_')
        patient = patient_parts[0]
        segmentation_path_pattern = os.path.join(directory, f"{patient}_*_GlistrBoost_out-labels.nii")
        segmentation_files = glob.glob(segmentation_path_pattern)
        if len(segmentation_files) == 0:
            print(f"Skipping DSIHE patient {patient_id} due to missing segmentation file.")
            return None
        if len(segmentation_files) > 1:
            print(f"Skipping DSIHE patient {patient_id} due to multiple segmentation files found.")
            return None
        data['segmentation'] = load_nifti_file(segmentation_files[0])

    elif dataset_type == 'brats_2017':
        modalities = ['flair', 't1', 't1ce', 't2']
        for modality in modalities:
            modality_path_pattern = os.path.join(directory, f"dsihe_Brats17_{patient_id}_{tumor_class}_{modality}.nii.gz")
            modality_files = glob.glob(modality_path_pattern)

            if len(modality_files) == 0:
                print(f"Skipping BRATS patient {patient_id} due to missing modality: {modality}")
                return None
            if len(modality_files) > 1:
                print(f"Skipping BRATS patient {patient_id} due to multiple files found for modality: {modality}")
                return None

            data[modality] = load_nifti_file(modality_files[0])

        segmentation_path = os.path.join(source_dir_brats_2017, tumor_class)
        segmentation_path_patient = os.path.join(segmentation_path, f"Brats17_{patient_id}")
        print(segmentation_path_patient)
        segmentation_path_pattern = os.path.join(segmentation_path_patient, f"Brats17_{patient_id}_seg.nii.gz")
        print(segmentation_path_pattern)
        segmentation_files = glob.glob(segmentation_path_pattern)
        if len(segmentation_files) == 0:
            print(f"Skipping BRATS patient {patient_id} due to missing segmentation file.")
            return None
        if len(segmentation_files) > 1:
            print(f"Skipping BRATS patient {patient_id} due to multiple segmentation files found.")
            return None

        # Convert the segmentation file from .nii to .nii.gz
        segmentation_data = load_nifti_file(segmentation_files[0])
        #segmentation_gz_path = segmentation_files[0]
        #save_nifti_file(segmentation_data, segmentation_gz_path)

        data['segmentation'] = segmentation_data

    elif dataset_type == 'brats_2019':
        modalities = ['flair', 't1', 't1ce', 't2']
        for modality in modalities:
            modality_path_pattern = os.path.join(directory, f"dsihe_BraTS19_{patient_id}_{tumor_class}_{modality}.nii.gz")
            modality_files = glob.glob(modality_path_pattern)

            if len(modality_files) == 0:
                print(f"Skipping BRATS patient {patient_id} due to missing modality: {modality}")
                return None
            if len(modality_files) > 1:
                print(f"Skipping BRATS patient {patient_id} due to multiple files found for modality: {modality}")
                return None

            data[modality] = load_nifti_file(modality_files[0])

        segmentation_path = os.path.join(source_dir_brats_2019, tumor_class)
        segmentation_path_patient = os.path.join(segmentation_path, f"BraTS19_{patient_id}")
        print(segmentation_path_patient)
        segmentation_path_pattern = os.path.join(segmentation_path_patient, f"BraTS19_{patient_id}_seg.nii.gz")
        print(segmentation_path_pattern)
        segmentation_files = glob.glob(segmentation_path_pattern)
        if len(segmentation_files) == 0:
            print(f"Skipping BRATS patient {patient_id} due to missing segmentation file.")
            return None
        if len(segmentation_files) > 1:
            print(f"Skipping BRATS patient {patient_id} due to multiple segmentation files found.")
            return None

        # Convert the segmentation file from .nii to .nii.gz
        segmentation_data = load_nifti_file(segmentation_files[0])
        #segmentation_gz_path = segmentation_files[0]
        #save_nifti_file(segmentation_data, segmentation_gz_path)

        data['segmentation'] = segmentation_data

    return data

# Function to split the dataset into training and test sets
def split_dataset(data, train_size=0.8):
    patient_ids = list(data.keys())
    train_ids, test_ids = train_test_split(patient_ids, test_size=(1 - train_size), random_state=42)

    train_set = {pid: data[pid] for pid in train_ids}
    test_set = {pid: data[pid] for pid in test_ids}

    return train_set, test_set

# Function to save the dataset
def save_dataset(dataset, output_dir):
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    for patient_id, modalities in dataset.items():
        patient_dir = os.path.join(output_dir, patient_id)
        if not os.path.exists(patient_dir):
            os.makedirs(patient_dir)

        for modality, data in modalities.items():
            file_path = os.path.join(patient_dir, f"{modality}.nii.gz")
            save_nifti_file(data, file_path)

# Run the extraction process for both datasets
traverse_and_extract(source_dir_brats_2017, brats_2017_dir, file_patterns_brats)
#traverse_and_extract(source_dir_brats_2019, brats_2019_dir, file_patterns_brats)

# Collect patient data from both datasets
'''patients_dsihe = []
for file in os.listdir(dsihe_dir):
    if file.endswith('.nii.gz'):
        parts = file.split('_')
        if len(parts) > 4:
            patient_id = '_'.join(parts[1:3])
            if patient_id not in patients_dsihe:
                patients_dsihe.append(patient_id)
'''
patients_brats_2017 = []
for file in os.listdir(brats_2017_dir):
    if file.endswith('.nii.gz'):
        parts = file.split('_')

        if len(parts) > 5:
            patient_id = '_'.join(parts[2:5])  # Extract the patient ID as 'YYYY_NN_N'
            print(parts)
            tumor_class = parts[5]  # Extract the tumor class (HGG/LGG)
            if patient_id not in patients_brats_2017:
                patients_brats_2017.append((patient_id, tumor_class))


# Load data for both datasets
data = {}
'''for patient_id in patients_dsihe:
    patient_data = get_patient_data(dsihe_dir, patient_id, 'dsihe')
    if patient_data:
        data[patient_id] = patient_data
        print(f"Loaded data for DSIHE patient: {patient_id}")'''

for patient_id, tumor_class in patients_brats_2017:
    patient_data = get_patient_data(brats_2017_dir, patient_id, 'brats_2017' ,tumor_class)
    if patient_data:
        data[f"{patient_id}_{tumor_class}"] = patient_data
        print(f"Loaded data for BRATS 2017 patient: {patient_id} with grade: {tumor_class}")


# Split the combined dataset
if len(data) == 0:
    print("No data available to split. Exiting...")
else:
    train_set, test_set = split_dataset(data)
    # Save the datasets
    save_dataset(train_set, os.path.join(output_dir, "train_set"))
    save_dataset(test_set, os.path.join(output_dir, "test_set"))


def count_folders(directory):
    if not os.path.exists(directory):
        return 0
    return len([name for name in os.listdir(directory) if os.path.isdir(os.path.join(directory, name))])

# Define the output directories
train_dir = os.path.join(output_dir, "train_set")
test_dir = os.path.join(output_dir, "test_set")

# Count the number of folders in each directory
train_count = count_folders(train_dir)
test_count = count_folders(test_dir)

# Print the counts
print(f"Number of folders in training set: {train_count}")
print(f"Number of folders in test set: {test_count}")

['dsihe', 'Brats17', 'TCIA', '479', '1', 'HGG', 'flair.nii.gz']
['dsihe', 'Brats17', 'TCIA', '479', '1', 'HGG', 't1.nii.gz']
['dsihe', 'Brats17', 'TCIA', '479', '1', 'HGG', 't1ce.nii.gz']
['dsihe', 'Brats17', 'TCIA', '479', '1', 'HGG', 't2.nii.gz']
['dsihe', 'Brats17', 'TCIA', '608', '1', 'HGG', 'flair.nii.gz']
['dsihe', 'Brats17', 'TCIA', '608', '1', 'HGG', 't1.nii.gz']
['dsihe', 'Brats17', 'TCIA', '608', '1', 'HGG', 't1ce.nii.gz']
['dsihe', 'Brats17', 'TCIA', '608', '1', 'HGG', 't2.nii.gz']
['dsihe', 'Brats17', 'TCIA', '603', '1', 'HGG', 'flair.nii.gz']
['dsihe', 'Brats17', 'TCIA', '603', '1', 'HGG', 't1.nii.gz']
['dsihe', 'Brats17', 'TCIA', '603', '1', 'HGG', 't1ce.nii.gz']
['dsihe', 'Brats17', 'TCIA', '603', '1', 'HGG', 't2.nii.gz']
['dsihe', 'Brats17', 'TCIA', '499', '1', 'HGG', 'flair.nii.gz']
['dsihe', 'Brats17', 'TCIA', '499', '1', 'HGG', 't1.nii.gz']
['dsihe', 'Brats17', 'TCIA', '499', '1', 'HGG', 't1ce.nii.gz']
['dsihe', 'Brats17', 'TCIA', '499', '1', 'HGG', 't2.nii.gz']
['ds